In [ ]:
import torch, torchvision
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch.nn.functional as F
from tqdm import tqdm

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import albumentations as A

def save_dataset(train_dir, mask_dir, save_dir, num_rep, transforms=None):
    train_dict = {p.stem: p for p in train_dir.iterdir()}
    mask_dict = {p.stem: p for p in mask_dir.iterdir()}

    ids = list(train_dict.keys())

    for id in tqdm(ids):
        # Get the original img, mask path
        img_path = train_dir / train_dict[id]
        mask_path = mask_dir / mask_dict[id]

        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        # Add img, mask to dataset
        train_list = [img]
        mask_list = [mask]

        if transforms != None:
            for i in range(num_rep): # How many copies of the image
                transformed = transforms(image=img, mask=mask)
                transformed_image = transformed['image']
                transformed_mask = transformed['mask']

                train_list.append(transformed_image)
                mask_list.append(transformed_mask)

        idx = 1
        for img, mask in zip(train_list, mask_list):
            name = f"{id}_{idx}.png"

            save_img_path = save_dir / "train" / name
            save_mask_path = save_dir / "train_label" / name

            cv2.imwrite(str(save_img_path), img)
            cv2.imwrite(str(save_mask_path), mask)

            idx += 1


In [ ]:
# Please refer to these adresses: https://albumentations.ai/docs/api_reference/augmentations/transforms/ and https://albumentations.ai/docs/api_reference/augmentations/geometric/transforms/

transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.ElasticTransform(),
    A.RandomRotate90(),
    A.CoarseDropout(max_holes = 10, max_height=16, max_width=16),
])

In [ ]:
save_dataset(train_dir = Path("/content/drive/MyDrive/Star/astrocyte_data/train"), mask_dir=Path("/content/drive/MyDrive/Star/astrocyte_data/train_label"),
             save_dir=Path("/content/drive/MyDrive/Star/astrocyte_aug"), num_rep=9, transforms=transforms)

100%|██████████| 23/23 [01:14<00:00,  3.24s/it]
